# RSNA-BASELINE-001 — Imaging Floor (Kaggle GPU)

**Hypothesis:** EfficientNet-B0 multi-plane mid-slice 2.5D on gold labels (~58) with StudyInstanceUID 5-fold CV establishes the OOF macro-AUC floor.

**Constraints:** No report at inference. No pseudo-labels / LLM labels / ensembles / HP search.

**Setup:** Attach competition `rsna-knee-abnormality-detection` + code Dataset with `src/` and `configs/`. GPU on. Never download DICOMs to a local Mac.

In [ ]:
import sys
from pathlib import Path

for p in [
    Path("/kaggle/input/rsna-knee-control/src"),
    Path("/kaggle/input/rsna-knee/src"),
    Path("/kaggle/working/src"),
]:
    if (p / "rsna_knee").exists():
        sys.path.insert(0, str(p))
        print("using", p)
        break
else:
    raise SystemExit("rsna_knee package not found — attach code Dataset")

In [ ]:
from rsna_knee.data.loaders import (
    load_series_metadata,
    load_test_metadata,
    load_train_metadata,
)
from rsna_knee.runtime.adapter import resolve_data_paths
from rsna_knee.training.imaging_loop import BaselineConfig, load_baseline_config, run_imaging_baseline
from rsna_knee.training.seed import seed_everything

seed_everything(42)
paths = resolve_data_paths(mode="kaggle", allow_missing_dicom=False)
print("root", paths.root)

config_path = None
for c in [
    Path("/kaggle/input/rsna-knee-control/configs/experiment/rsna_baseline_001.yaml"),
    Path("/kaggle/input/rsna-knee/configs/experiment/rsna_baseline_001.yaml"),
]:
    if c.exists():
        config_path = c
        break

cfg = load_baseline_config(config_path) if config_path else BaselineConfig()
# cfg.pilot_folds = 1  # optional 1-fold I/O pilot before full 5-fold

result = run_imaging_baseline(
    load_train_metadata(paths.train_csv),
    load_series_metadata(paths.train_series_csv),
    load_test_metadata(paths.test_csv),
    load_series_metadata(paths.test_series_csv),
    train_series_root=paths.train_series_dir,
    test_series_root=paths.test_series_dir,
    output_dir=paths.working_dir / cfg.experiment_id,
    cfg=cfg,
    config_path=config_path,
)

sub_src = Path(result.output_dir) / "submission.csv"
(paths.working_dir / "submission.csv").write_bytes(sub_src.read_bytes())

print("OOF macro AUC", result.oof_macro_auc)
print("per-label", result.per_label_auc)
print("train_s", result.train_runtime_sec, "infer_s", result.infer_runtime_sec)
print("rss_mb", result.peak_rss_mb, "cuda_mb", result.cuda_peak_mb)
print(result.conclusion)